In [1]:
library(googledrive)
drive_auth()

Is it OK to cache OAuth access credentials in the folder ~/.cache/gargle
between R sessions?
1: Yes
2: No


Selection: 1


Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=6f62e83af6f4c098bb8beb8367d4a548&access_type=offline&prompt=consent



Enter authorization code: eyJjb2RlIjoiNC8wQWRrVkxQemROS3hsZEpVbDJvZEVBMFpjcDVrZ05nbDBNZHc3cHhsdFFFVEhFbXhpS2NMRjk4SVg5S203ZEk0dGE3R29KQSIsInN0YXRlIjoiNmY2MmU4M2FmNmY0YzA5OGJiOGJlYjgzNjdkNGE1NDgifQ==


In [2]:
system("sudo apt-get install libgmp-dev")
system("sudo apt-get install libmagick++-dev")
system("sudo apt-get install libgsl-dev")
system("sudo apt-get install libmpfr-dev")
system("sudo apt-get install libgtk-3-dev libcairo2-dev")

.required_pkgs <- c(
  "EnvStats", "psych", "expm",
  "Rfast", "foreach", "doParallel", "devtools"
)
for (.pkg in .required_pkgs) {
  if (!requireNamespace(.pkg, quietly = TRUE)) {
    if (.pkg == "Rfast") {
      install.packages(.pkg, INSTALL_opts = "--no-lock")
    } else {
      install.packages(.pkg)
    }
  }
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘nortest’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘mnormt’, ‘GPArotation’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘zigg’, ‘RcppParallel’, ‘RcppArmadillo’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘iterators’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
library(devtools)
install.packages("remotes")

Loading required package: usethis

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [4]:
##install.packages("/content/rdetools_1.0.tar.gz", repos = NULL, type="source")
# Instala rrcov desde la version local modificada (1.7-7.9000).
# Solo reinstala si la version instalada es distinta a la local.

  devtools::install_local("/content/rrcov.zip", upgrade = "never", quiet = TRUE)



Warning message:
“`install_local()` was deprecated in devtools 2.5.0.
ℹ Please use pak::pak("local::path") instead.”
Installing 4 packages: mvtnorm, DEoptimR, pcaPP, robustbase



In [5]:
library(MASS)
library(rrcov)
library(EnvStats)
library(psych)
library(expm)
library(Rfast)
library(foreach)
library(doParallel)

Loading required package: robustbase

Scalable Robust Estimators with High Breakdown Point (version 1.7-7)



Attaching package: ‘EnvStats’


The following object is masked from ‘package:rrcov’:

    predict


The following object is masked from ‘package:MASS’:

    boxcox


The following objects are masked from ‘package:stats’:

    predict, predict.lm


The following object is masked from ‘package:base’:

    print.default


Loading required package: Matrix


Attaching package: ‘expm’


The following object is masked from ‘package:Matrix’:

    expm


The following object is masked from ‘package:rrcov’:

    sqrtm


Loading required package: Rcpp

Loading required package: zigg

Loading required package: RcppParallel


Attaching package: ‘RcppParallel’


The following object is masked from ‘package:Rcpp’:

    LdFlags



Rfast: 2.1.5.2

 ___ __ __ __ __    __ __ __ __ __ _             _               __ __ __ __ __     __ __ __ __ __ __   
|  __ __ __ __  |  |  __ __ __ __ _/        

In [6]:
MethodUCLKernel <- function(T2, alpha)
{
  EstKernelSmooth = density(T2,kernel="gaussian",bw="nrd")
  valuePairs = cbind(EstKernelSmooth$x,EstKernelSmooth$y)
  valuePairs = valuePairs[valuePairs[,1]>=0,]
  Estand = valuePairs[,2]/sum(valuePairs[,2])
  NumberRow = nrow(valuePairs)
  countValues = 0

  for(i in 1:NumberRow)
  {
    if(countValues <  (1-alpha))
    {
      j = i
      countValues = sum(Estand[1:i])
    }
  }

  UCL = valuePairs[j,1]
  return(list(UCL=UCL,alpha1=(1-countValues)))
}

SignalProbability <- function (matrixT2, ucl, uclMax, uclKernel )
{

  SignalCountUCL = 0
  SignalCountMax = 0
  SignalCountKernel = 0

  for (i in 1: dim(matrixT2)[1])
  {
    RowMatrix = matrixT2[i,]

    ListFilterUCL = Filter(function(x) x > ucl,RowMatrix)
    SignalCountUCL = SignalCountUCL + length(ListFilterUCL)


    ListFilterMax = Filter(function(x) x > uclMax,RowMatrix)
    SignalCountMax = SignalCountMax + length(ListFilterMax)

    ListFilterKernel = Filter(function(x) x > uclKernel,RowMatrix)
    SignalCountKernel = SignalCountKernel + length(ListFilterKernel)
  }

  TotalValue = dim(matrixT2)[1]*dim(matrixT2)[2]

  SignalPro = SignalCountUCL/TotalValue
  SignalProMax = SignalCountMax/TotalValue
  SignalProKernel = SignalCountKernel/TotalValue

  return(list("SignalPro" = SignalPro,
              "SignalProMax" = SignalProMax,
              "SignalProKernel" = SignalProKernel))
}

AlgorithmMDPCFPart1 <- function(DatesNorm, NumVariable, Observations, Alpha = 0.05, outliersData = c(), OutliersFlag = FALSE) {
  MiddlePoint <- floor(Observations / 2) + 1
  AlgorithmObjects <- AlgorithmRoMDP(DatesNorm)
  # calculo de Media y matrices de correlacion y de varianza
  MeanDMP <- unlist(AlgorithmObjects[1])
  SigmaDMP <- matrix(diag(as.numeric(unlist(AlgorithmObjects[2]))), ncol = NumVariable)
  InverseSigmaDMP <- solve(SigmaDMP)
  SigmaOfMinimunDet <- AlgorithmObjects[3][[1]]
  CorMatrix <- sqrt(InverseSigmaDMP) %*% SigmaOfMinimunDet %*% sqrt(InverseSigmaDMP)

  # Estimacion objetos algoritmo Ebadi
  TraceRhoSquare <- tr(CorMatrix %^% 2) - (NumVariable**2) / MiddlePoint
  TraceRhoCubic <- tr(CorMatrix %^% 3) - ((3 * NumVariable) / MiddlePoint) * tr(CorMatrix %^% 2) + ((2 * (NumVariable**3)) / (MiddlePoint**2))

  # Estimadores Ui y Zi
  ListEstimUi <- c()
  ListEstimZi <- c()
  ListOutliers <- c()
  Zalpha <- qnorm(1 - Alpha, 0, 1)
  ConstantMDP <- 1 + (2 * NumVariable) / (Observations * sqrt(TraceRhoSquare))


  if (OutliersFlag) {
    for (i in 1:dim(outliersData)[1])
    {
      DistanceMahalanobisXi <- t((outliersData[i, ] - MeanDMP)) %*% InverseSigmaDMP %*% (outliersData[i, ] - MeanDMP)
      Ui <- (DistanceMahalanobisXi - NumVariable) / (2 * ConstantMDP * sqrt(TraceRhoSquare))
      ListEstimUi <- c(ListEstimUi, Ui)
      Zi <- Ui - (4 * TraceRhoCubic * (Zalpha**2 - 1)) / (3 * (2 * TraceRhoSquare)^(3 / 2))
      ListEstimZi <- c(ListEstimZi, Zi)
    }
  } else {
    for (i in 1:dim(DatesNorm)[1])
    {
      DistanceMahalanobisXi <- t((DatesNorm[i, ] - MeanDMP)) %*% InverseSigmaDMP %*% (DatesNorm[i, ] - MeanDMP)
      Ui <- (DistanceMahalanobisXi - NumVariable) / (2 * ConstantMDP * sqrt(TraceRhoSquare))
      ListEstimUi <- c(ListEstimUi, Ui)
      Zi <- Ui - (4 * TraceRhoCubic * (Zalpha**2 - 1)) / (3 * (2 * TraceRhoSquare)^(3 / 2))
      ListEstimZi <- c(ListEstimZi, Zi)
    }
  }
  return(list("Ui" = ListEstimUi, "Zi" = ListEstimZi))
}

AlgorithmRoMDP <- function(DataSet, itertime = 100) {
    SampleNumber <- dim(DataSet)[1]
    VariableNumber <- dim(DataSet)[2]
    MiddlePoint <- round(SampleNumber / 2) + 1
    TransposedDataSet <- t(DataSet)

    ValueInitialSubsets <- 2
    BestDet <- 0
    VecZero <- numeric(SampleNumber)

    # Ciclo de iteracciones por default 100
    for (iter in 1:itertime) {
        Id <- sample(SampleNumber, ValueInitialSubsets, replace = FALSE)
        SubsetById <- DataSet[Id, ]
        MuSubsetById <- Rfast::colmeans(SubsetById)
        VarSubsetById <- Rfast::colVars(SubsetById)
        Sama <- (TransposedDataSet - MuSubsetById) / VarSubsetById
        Distance <- Rfast::colsums(Sama)
        Criterio <- 10
        Count <- 0
        while (Criterio != 0 & Count <= 15) {
            Count <- Count + 1
            VectorZeroi <- numeric(SampleNumber)
            DistancePerm <- order(Distance)
            VectorZeroi[DistancePerm[1:MiddlePoint]] <- 1
            Criterio <- sum(abs(VectorZeroi - VecZero))
            VecZero <- VectorZeroi
            NewDataSet <- DataSet[DistancePerm[1:MiddlePoint], ]
            MuSubsetById <- Rfast::colmeans(NewDataSet)
            VarSubsetById <- Rfast::colVars(NewDataSet)
            Sama <- (TransposedDataSet - MuSubsetById) / VarSubsetById
            Distance <- Rfast::colsums(Sama)
        }
        TempDet <- prod(VarSubsetById)
        if (BestDet == 0 | TempDet < BestDet) {
            BestDet <- TempDet
            FinalVec <- VecZero
        }
    }
    SubMCD <- (1:SampleNumber)[FinalVec != 0]

    MuSubsetById <- Rfast::colmeans(DataSet[SubMCD, ])
    VarSubsetById <- Rfast::colVars(DataSet[SubMCD, ])
    Sigma <- cov(DataSet[SubMCD, ])
    return(list(MuSubsetById, VarSubsetById, Sigma, SubMCD))
}

mahalanobis_tpu_batch <- function(x_matrix, center, cov_inv, use_tpu = FALSE) {
  return(mahalanobis(x_matrix, center, cov_inv))
}

# ---- Copula Gaussiana: genera datos Gamma multivariados ----
# Marginals: X_d ~ Gamma(shape_d, rate_d)
# Dependence: copula con matriz de correlacion corr
rmvgamma <- function(n, shape = 1, rate = 1, corr = diag(length(shape))) {
  if (!is.matrix(corr) || !isSymmetric(corr))
    stop("'corr' must be a symmetric matrix")
  D     <- ncol(corr)
  shape <- rep(shape, length.out = D)
  rate  <- rep(rate,  length.out = D)
  if (D == 1L) return(rgamma(n, shape, rate))
  Z   <- MASS::mvrnorm(n, mu = rep(0, D), Sigma = corr)
  cdf <- pnorm(Z)
  sapply(1:D, function(d) qgamma(cdf[, d], shape[d], rate[d]))
}

# ---- Simulacion Fase I bajo control (distribucion Gamma) ----
SimulationT2Chart <- function(observation, numVariables, numSimulation,
                               sigmaMatriz, rho, alphaMRCD = 0.75,
                               typeMethod = "MRCD", beta = 1) {
  T2Total   <- c()
  T2Max     <- c()
  LenMatrix <- observation

  for (i in 1:numSimulation) {
    Data <- rmvgamma(observation, rho, beta, sigmaMatriz)

    if (typeMethod == "MRCD") {
      CovMRCD        <- CovMrcd(Data, alpha = alphaMRCD)
      MediaMRCD      <- CovMRCD$center
      SigmaMRCD      <- CovMRCD$cov
      BestSubset     <- CovMRCD$best
      DataBestSubset <- Data[BestSubset, ]
      T2             <- mahalanobis_tpu_batch(DataBestSubset, center = MediaMRCD, cov = SigmaMRCD)
      LenMatrix      <- length(BestSubset)

    } else if (typeMethod == "T2MOD") {
      Media    <- colMeans(Data)
      Sigma    <- cov(Data)
      SigmaMod <- 1 / (sum(diag(Sigma)) / numVariables)
      T2 <- c()
      for (j in 1:dim(Data)[1]) {
        Ai <- (observation / (observation - 1)) *
              (norm((Data[j, ] - Media) / sqrt(numVariables)^2, type = "2") / SigmaMod)
        T2 <- c(Ai, T2)
      }
      LenMatrix <- observation

    } else if (typeMethod == "EBADIUI") {
      Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, FALSE)
      T2        <- Firstestimation$Ui
      LenMatrix <- observation

    } else if (typeMethod == "EBADIZI") {
      Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, FALSE)
      T2        <- Firstestimation$Zi
      LenMatrix <- observation
    }

    T2Total <- c(T2Total, T2)
    T2Max   <- c(T2Max, max(T2))
  }

  T2Matrix <- matrix(T2Total, ncol = LenMatrix)
  return(list("T2Matrix" = T2Matrix, "T2Total" = T2Total, "T2Max" = T2Max))
}

# ---- Simulacion Fase I con valores atipicos (distribucion Gamma) ----
# rhoShift1: escalar de desplazamiento (0 = sin atipicos)
SimulationT2ChartOutliers <- function(observation, numVariables, numSimulation,
                                       sigmaMatriz, rho, rhoShift,
                                       percentoutliers = 0, alphaMRCD = 0.75,
                                       DeltaNCP = 0.05,
                                       UCL = 1, UCLMax = 1, UCLKernel = 1,
                                       typeMethod = "MRCD", beta = 1,
                                       rhoShift1 = 0) {
  T2Total     <- c()
  T2Max       <- c()
  NumOutliers <- floor(percentoutliers * observation)

  for (i in 1:numSimulation) {
    if (rhoShift1 == 0) {
      Data <- rmvgamma(observation, rho, beta, sigmaMatriz)
    } else {
      Data <- rmvgamma(observation - NumOutliers, rho, beta, sigmaMatriz)
      if (NumOutliers >= 1) {
        DataOutlier <- rmvgamma(NumOutliers, rhoShift, beta, sigmaMatriz)
        Data        <- rbind(Data, DataOutlier)
      }
    }

    if (typeMethod == "MRCD") {
      CovMRCD        <- CovMrcd(Data, alpha = alphaMRCD)
      MediaMRCD      <- CovMRCD$center
      SigmaMRCD      <- CovMRCD$cov
      BestSubset     <- CovMRCD$best
      DataBestSubset <- Data[BestSubset, ]
      if (rhoShift1 == 0) {
        T2 <- mahalanobis_tpu_batch(DataBestSubset, center = MediaMRCD, cov = SigmaMRCD)
      } else {
        T2 <- mahalanobis_tpu_batch(DataOutlier,    center = MediaMRCD, cov = SigmaMRCD)
      }

    } else if (typeMethod == "T2MOD") {
      Media    <- colMeans(Data)
      Sigma    <- cov(Data)
      SigmaMod <- 1 / (sum(diag(Sigma)) / numVariables)
      target   <- if (rhoShift1 == 0) Data else DataOutlier
      T2 <- c()
      for (j in 1:dim(target)[1]) {
        Ai <- (observation / (observation - 1)) *
              (norm((target[j, ] - Media) / sqrt(numVariables)^2, type = "2") / SigmaMod)
        T2 <- c(Ai, T2)
      }

    } else if (typeMethod == "EBADIUI") {
      if (rhoShift1 == 0) {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, c(), FALSE)
        T2 <- Firstestimation$Ui
      } else {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, DataOutlier, TRUE)
        T2 <- Firstestimation$Ui
      }

    } else if (typeMethod == "EBADIZI") {
      if (rhoShift1 == 0) {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, c(), FALSE)
        T2 <- Firstestimation$Zi
      } else {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, DataOutlier, TRUE)
        T2 <- Firstestimation$Zi
      }
    }

    T2Total <- c(T2Total, T2)
    T2Max   <- c(T2Max, max(T2))
  }

  T2Matrix <- matrix(T2Total, ncol = 1)
  SignalProbabilityT2OutliersMRCD <- SignalProbability(T2Matrix, UCL, UCLMax, UCLKernel)
  return(c(
    DeltaNCP,
    SignalProbabilityT2OutliersMRCD$SignalPro,
    SignalProbabilityT2OutliersMRCD$SignalProMax,
    SignalProbabilityT2OutliersMRCD$SignalProKernel
  ))
}

In [ ]:
n_cores_available <- parallel::detectCores()
n_cores_use <- max(1L, n_cores_available - 1L)
cl <- makeCluster(n_cores_use)
registerDoParallel(cl)

set.seed(123)

for(NumberVariable in c(200, 250)){
  Observation        <- 150
  Percentoutliers    <- 0.1
  NumSimulation      <- 10000
  AlphaMRCD          <- 0.75
  AlphaPFA           <- 0.05
  NumSimulationDelta <- 10000

  # Parametros Gamma — Escenario 1: shape=rep(1:5,...), rate=rep(c(1,3,0.5,2,5),...)
  # E[X_d] = shape_d / rate_d,  Var[X_d] = shape_d / rate_d^2
  rho  <- rep(2,             times = NumberVariable)
  beta <- rep(5, times = NumberVariable)

  path <- paste0("/content/SinalprobabilityGamma(2,5)", Observation,
                 "x", NumberVariable, "x", Percentoutliers,
                 "x", NumSimulation, ".RData")

  fun <- function(i, j) (0.5)^(abs(i - j))
  SigmaCorr <- outer(1:NumberVariable, 1:NumberVariable, FUN = fun)
  Inverse   <- solve(SigmaCorr)

  NumBatches       <- 10L
  SimPerBatch      <- NumSimulation      %/% NumBatches
  SimPerBatchDelta <- NumSimulationDelta %/% NumBatches

  # ── Parte 1: Limites de control (UCL) ─────────────────────────────────────
  .local_fns <- c("SimulationT2Chart", "rmvgamma",
                  "mahalanobis_tpu_batch", "AlgorithmMDPCFPart1", "AlgorithmRoMDP")
  .pkgs <- c("rrcov", "MASS", "Rfast", "EnvStats", "KernSmooth", "psych", "expm")

  .methods1 <- c("MRCD", "T2MOD", "EBADIUI", "EBADIZI")
  .p1_grid  <- expand.grid(method = .methods1, batch = seq_len(NumBatches),
                            stringsAsFactors = FALSE)

  .p1_raw <- foreach(
    ti            = seq_len(nrow(.p1_grid)),
    .combine      = "list",
    .multicombine = TRUE,
    .packages     = .pkgs,
    .export       = c(.local_fns, ".p1_grid", "SimPerBatch", "AlphaMRCD",
                      "rho", "beta", "SigmaCorr", "Observation", "NumberVariable")
  ) %dopar% {
    res <- SimulationT2Chart(
      Observation, NumberVariable, SimPerBatch,
      SigmaCorr, rho, AlphaMRCD, .p1_grid$method[ti], beta
    )
    list(method    = .p1_grid$method[ti],
         T2Total   = res$T2Total,
         T2Max     = res$T2Max,
         LenMatrix = ncol(res$T2Matrix))
  }

  # Combinar lotes por metodo
  valuesT2Total <- matrix(vector("list", 3L * 4L), nrow = 3L, ncol = 4L)
  for (.m_i in seq_along(.methods1)) {
    .m       <- .methods1[.m_i]
    .idx     <- which(sapply(.p1_raw, `[[`, "method") == .m)
    .T2Total <- unlist(lapply(.p1_raw[.idx], `[[`, "T2Total"))
    .T2Max   <- unlist(lapply(.p1_raw[.idx], `[[`, "T2Max"))
    .Len     <- .p1_raw[[.idx[1L]]]$LenMatrix
    valuesT2Total[[1L, .m_i]] <- matrix(.T2Total, ncol = .Len)
    valuesT2Total[[2L, .m_i]] <- .T2Total
    valuesT2Total[[3L, .m_i]] <- .T2Max
  }

  # UCL por metodo
  ValuesMRCDT2Matrix <- valuesT2Total[1, 1]
  ValuesMRCDT2Total  <- valuesT2Total[2, 1]
  ValuesMRCDT2Max    <- valuesT2Total[3, 1]
  KernelMethodMRCD   <- MethodUCLKernel(ValuesMRCDT2Total[[1]], AlphaPFA)
  UCLMRCD            <- qemp(p = 1 - AlphaPFA, obs = ValuesMRCDT2Total[[1]])
  UCLMaxMRCD         <- qemp(p = 1 - AlphaPFA, obs = ValuesMRCDT2Max[[1]])
  UCLKernelMRCD      <- KernelMethodMRCD$UCL
  SignalProbabilityT2MRCD <- SignalProbability(ValuesMRCDT2Matrix[[1]], UCLMRCD, UCLMaxMRCD, UCLKernelMRCD)

  ValuesT2MODT2Matrix <- valuesT2Total[1, 2]
  ValuesT2MODT2Total  <- valuesT2Total[2, 2]
  ValuesT2MODT2Max    <- valuesT2Total[3, 2]
  KernelMethodT2MOD   <- MethodUCLKernel(ValuesT2MODT2Total[[1]], AlphaPFA)
  UCLT2MOD            <- qemp(p = 1 - AlphaPFA, obs = ValuesT2MODT2Total[[1]])
  UCLMaxT2MOD         <- qemp(p = 1 - AlphaPFA, obs = ValuesT2MODT2Max[[1]])
  UCLKernelT2MOD      <- KernelMethodT2MOD$UCL
  SignalProbabilityT2MOD <- SignalProbability(ValuesT2MODT2Matrix[[1]], UCLT2MOD, UCLMaxT2MOD, UCLKernelT2MOD)

  ValuesEBADIZIT2Matrix <- valuesT2Total[1, 4]
  ValuesEBADIZIT2Total  <- valuesT2Total[2, 4]
  ValuesEBADIZIT2Max    <- valuesT2Total[3, 4]
  KernelMethodEBADIZI   <- MethodUCLKernel(ValuesEBADIZIT2Total[[1]], AlphaPFA)
  UCLEBADIZI            <- qemp(p = 1 - AlphaPFA, obs = ValuesEBADIZIT2Total[[1]])
  UCLMaxEBADIZI         <- qemp(p = 1 - AlphaPFA, obs = ValuesEBADIZIT2Max[[1]])
  UCLKernelEBADIZI      <- KernelMethodEBADIZI$UCL
  SignalProbabilityT2EBADIZI <- SignalProbability(ValuesEBADIZIT2Matrix[[1]], UCLEBADIZI, UCLMaxEBADIZI, UCLKernelEBADIZI)

  # ── Parte 2: Senal de probabilidad por delta ─────────────────────────────
  # Desplazamiento escalar s sobre shape: rho_shift = rho + s
  # E[X_shift] - E[X_0] = s / beta  =>  delta = sqrt((s/beta)' Sigma^-1 (s/beta))
  ArrayRhoShift <- c(0, 1/100, 5/100, 10/100, 15/100, 25/100, 50/100, 60/100, 75/100, 90/100, 1)

  .ucl_map <- list(
    MRCD    = c(UCLMRCD,    UCLMaxMRCD,    UCLKernelMRCD),
    T2MOD   = c(UCLT2MOD,   UCLMaxT2MOD,   UCLKernelT2MOD),
    EBADIZI = c(UCLEBADIZI, UCLMaxEBADIZI, UCLKernelEBADIZI)
  )
  .delta_methods <- c("MRCD", "T2MOD", "EBADIZI")
  .n_shifts  <- length(ArrayRhoShift)
  .n_methods <- length(.delta_methods)
  .n_ms      <- .n_methods * .n_shifts
  .n_tasks_d <- .n_ms * NumBatches

  .delta_fns <- c("SimulationT2ChartOutliers", "rmvgamma",
                  "AlgorithmMDPCFPart1", "AlgorithmRoMDP",
                  "mahalanobis_tpu_batch", "SignalProbability")

  .delta_raw <- foreach(
    task_idx  = seq_len(.n_tasks_d),
    .combine  = "rbind",
    .packages = .pkgs,
    .export   = c(.delta_fns,
                  "ArrayRhoShift", ".ucl_map", ".delta_methods",
                  ".n_shifts", ".n_ms",
                  "Observation", "NumberVariable", "SimPerBatchDelta",
                  "rho", "beta", "SigmaCorr", "Percentoutliers", "AlphaMRCD", "Inverse")
  ) %dopar% {
    ms_idx  <- ((task_idx - 1L) %% .n_ms) + 1L
    m_idx   <- ((ms_idx   - 1L) %/% .n_shifts) + 1L
    s_idx   <- ((ms_idx   - 1L) %% .n_shifts) + 1L
    typeM   <- .delta_methods[m_idx]
    shiftmu <- ArrayRhoShift[s_idx]          # escalar
    rhosum  <- rho + shiftmu                 # vector shape desplazado
    ucls    <- .ucl_map[[typeM]]

    # delta = ||E[X_shift] - E[X_0]||_{Sigma^-1}
    # E[Gamma(rho,beta)] = rho/beta  =>  diferencia = shiftmu/beta
    mu_diff  <- shiftmu / beta
    DeltaNCP <- sqrt(as.numeric(t(mu_diff) %*% Inverse %*% mu_diff))

    SimulationT2ChartOutliers(
      Observation, NumberVariable, SimPerBatchDelta,
      SigmaCorr, rho, rhosum, Percentoutliers, AlphaMRCD, DeltaNCP,
      ucls[1], ucls[2], ucls[3], typeM, beta, shiftmu
    )
  }

  # Promediar probabilidades de senal entre lotes
  .delta_all <- matrix(0, .n_ms, 4L)
  for (.ms_i in seq_len(.n_ms)) {
    .rows <- .delta_raw[seq(.ms_i, .n_tasks_d, by = .n_ms), , drop = FALSE]
    .delta_all[.ms_i, ] <- c(.rows[1L, 1L], colMeans(.rows[, 2:4, drop = FALSE]))
  }

  MatrixDeltaMRCD    <- .delta_all[seq_len(.n_shifts), ]
  MatrixDeltaT2MOD   <- .delta_all[seq_len(.n_shifts) + .n_shifts, ]
  MatrixDeltaEBADIZI <- .delta_all[seq_len(.n_shifts) + 2L * .n_shifts, ]

  save.image(path)
  drive_upload(path, path = "Colab10000Gamma/")
}

stopCluster(cl)

Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): SimulationT2Chart, rmvgamma, mahalanobis_tpu_batch, AlgorithmMDPCFPart1, AlgorithmRoMDP, SimPerBatch, AlphaMRCD, rho, beta, SigmaCorr, Observation, NumberVariable”
Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): SimulationT2ChartOutliers, rmvgamma, AlgorithmMDPCFPart1, AlgorithmRoMDP, mahalanobis_tpu_batch, SignalProbability, ArrayRhoShift, Observation, NumberVariable, SimPerBatchDelta, rho, beta, SigmaCorr, Percentoutliers, AlphaMRCD, Inverse”


In [ ]:
system("kill -9 -1")